# ImprovedGS cho public scene HCM0204

Notebook này chạy trọn pipeline VAI: preprocess, train ImprovedGS, render test poses, đánh giá ground truth và đóng gói kết quả.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

PHASE_DIR = Path('/kaggle/input/datasets/xuanph/phase1/phase1')
PUBLIC_SET = PHASE_DIR / 'public_set'
WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'Improved-GS'
PUBLISH_BRANCH = 'agent/vai-kaggle-render'

assert (PUBLIC_SET / 'HCM0204').is_dir(), f'Khong tim thay HCM0204: {PUBLIC_SET}'

In [ ]:
# Clone code chinh cua su phu.
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--recursive',
        'https://github.com/mdd206/Improved-GS.git', str(REPO_DIR)
    ], check=True)
if not (REPO_DIR / 'vai_render.py').is_file():
    subprocess.run(['git', 'fetch', 'origin', PUBLISH_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', PUBLISH_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'submodule', 'update', '--init', '--recursive'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print(Path.cwd())

In [ ]:
# Cai dependency Python va CUDA extension.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy==1.26.1', 'opencv-python==4.10.0.82',
                'setuptools==69.5.1', 'tqdm', 'plyfile'], check=True)
for package_dir in [
    'submodules/diff-gaussian-rasterization',
    'submodules/simple-knn',
    'submodules/fused-ssim',
]:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    package_dir, '--no-build-isolation'], check=True)
if shutil.which('colmap') is None:
    raise RuntimeError('Kaggle image chua co COLMAP CLI trong PATH')

In [ ]:
# Chuyen SIMPLE_RADIAL thanh PINHOLE RGBA cho HCM0204.
subprocess.run([
    sys.executable, 'vai_preprocess.py',
    '--input', str(PUBLIC_SET),
    '--output', str(WORK_ROOT / 'vai_cleaned'),
    '--subset', 'HCM0204',
], check=True)

In [ ]:
# In command de kiem tra path va tham so truoc khi train.
subprocess.run([
    sys.executable, 'run.py',
    '-c', 'configs/vai_hcm0204.json',
    '--dry_run',
], check=True)

In [ ]:
# Train ImprovedGS, render 60 test pose va tinh metric public.
env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.run([
    sys.executable, 'run.py',
    '-c', 'configs/vai_hcm0204.json',
], check=True, env=env)

In [ ]:
# Validate ten, kich thuoc va tao ZIP render.
subprocess.run([
    sys.executable, 'vai_package.py',
    '--phase_dir', str(PHASE_DIR),
    '--set_name', 'public_set',
    '--submission_dir', str(WORK_ROOT / 'vai_renders'),
    '--zip_path', str(WORK_ROOT / 'HCM0204_render.zip'),
    '--subset', 'HCM0204',
    '--output_extension', 'csv',
], check=True)

# Dong goi JSON evaluation rieng de tai ve.
shutil.make_archive(
    str(WORK_ROOT / 'HCM0204_eval'),
    'zip',
    root_dir=WORK_ROOT / 'vai_eval',
)
print(WORK_ROOT / 'HCM0204_render.zip')
print(WORK_ROOT / 'HCM0204_eval.zip')